# NBA 3.1 — Cohort Generation

Builds the **28 NBA 3.1 cohorts** (c1–c28) for each feature family and saves them to the 3.1 Volume. Feature notebooks then read the saved cohorts through one small API, `CohortGenerator`, and never rebuild them.

The schedule follows `Extra Notebooks/NBA_3.1_Model_Architecture_and_Cohorts.xlsx` (the source of truth).

**Inputs** (Excel "Inputs" section)
* c1 prediction month: `COHORT_1_PREDICTION_MONTH = 2024-07-01`
* `LOOKBACK_WINDOW_MONTHS = 12`
* `DATA_LAG_MONTHS = 1` (4-week lag for claims)
* Cohorts: c1–c24 Training, c25–c26 Testing, c27–c28 Holdout (Backtesting), 28 in total

**Feature families and offsets**

| Family | Features | Constant | Offset |
|:--|:--|:--|--:|
| `claims` | Rx, Dx, Px, Lab (4-week lag, month boundaries) | `CLAIMS_ANCHOR_OFFSET_DAYS` | 0 |
| `promotional` | Calls, iRep, DTC, Digital, Samples, Speaker, FTP, HQ Emails, ClickStream (2-week lag, day 17) | `PROMOTIONAL_ANCHOR_OFFSET_DAYS` | 17 |
| `mmit` | MMIT | `MMIT_ANCHOR_OFFSET_DAYS` | None (TBD) |
| `laad_lrx` | LAAD-based LRx | `LAAD_LRX_ANCHOR_OFFSET_DAYS` | None (TBD) |

While an offset is `None`, `CohortGenerator('<family>')` raises a `ValueError` naming the constant, and nothing is saved for that family.

**Per-cohort fields** (cohort i = 1..28)
* `cohort_id` = `c<i>`
* `prediction_month` = c1 prediction month + (i − 1) months, the 1st of the month
* `dataset` = Training (i ≤ 24), Testing (25–26), Holdout (Backtesting) (27–28)
* `offset_days` = the family's offset
* `lookback_start_date`, `lookback_end_date` as follows

**Claims** (offset 0, month granularity)
* `lookback_end_date` = prediction month − (`DATA_LAG_MONTHS` + 1) months, on day 1
* `lookback_start_date` = `lookback_end_date` − (`LOOKBACK_WINDOW_MONTHS` − 1) months, on day 1
* c1: prediction 2024-07-01, lookback 2023-06-01 to 2024-05-01

**Promotional** (offset 17, day-17 granularity)
* `lookback_end_date` = day 17 of the month before the prediction month
* `lookback_start_date` = `lookback_end_date` − `LOOKBACK_WINDOW_MONTHS` months, same day
* c1: prediction 2024-07-01, lookback 2023-06-17 to 2024-06-17

**MMIT / LAAD LRx:** the formulas will be added once their offsets are chosen.

**Saved files** under `/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/cohorts/`
* `<family>/cohorts.parquet`: the 28 cohorts
* `<family>/metadata.json`: offset_days, generation_timestamp, cohort_count, lookback_window_months, data_lag_months

Only `claims/` and `promotional/` are written while the MMIT and LAAD LRx offsets are `None`.

**Validation before saving:** exactly 28 rows; `cohort_id` = c1..c28 in order; 24 Training + 2 Testing + 2 Holdout rows; c1's lookback dates match the Excel (claims 2023-06-01 → 2024-05-01, promotional 2023-06-17 → 2024-06-17). Any failed check raises with the check that failed.

**Running vs importing**
* **Run this notebook directly** (or as a job task): it tries all four families. It prints `SKIPPED <family> — <CONSTANT> is None` or `GENERATED <family> — offset=…, rows=…, path=…`. Nothing else is saved.
* **`%run` it from a feature notebook:** only the definitions load, and nothing is regenerated.

```python
%run ../00_Helper_Notebooks/00_cohort_generation

# In a claims feature notebook:
claims_cohorts = CohortGenerator('claims').load()

# In a promotional feature notebook:
promo_cohorts = CohortGenerator('promotional').load()
```

To rebuild one family on purpose: `g = CohortGenerator('claims'); g.save(g.generate())`.

The base `00_cohort_config` is still `%run` below, so its primitives (`get_cohort_bounds`, `get_cohort_months`, `assign_cohort_month`) stay available to callers.

**Convention change**

The base primitives `get_cohort_bounds`, `get_cohort_months`, `assign_cohort_month` now default `offset_days=0` (NBA 3.1 claims convention). The NBA 3.0 default of 7 is removed. Callers that need a non-zero offset must pass it explicitly — use `CohortGenerator(family).offset_days` to get the right value for your feature family.

(Implementation: the base notebook is unchanged. The cell right after its `%run` re-binds the three functions with `offset_days=0` as the default; an explicit `offset_days=None` also means 0. Other base globals built at load time with offset 7, such as `df_cohort_ref`, `COHORT_ID`, `CUTOFF_DATE`, `WINDOW_BOUNDS` and `_cohort_month_sql()`, are not covered by this change.)

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/03_Feature_Engineering/03_HCP_level_features/00_cohort_config"

In [0]:
# ============================================================
# NBA 3.1 CONVENTION: the base primitives default to offset_days = 0 (claims), not the
# NBA 3.0 value 7. The primitives are defined in the base 00_cohort_config (%run above),
# where offset_days=None falls back to its global OFFSET_DAY = 7. Here they are re-bound
# with offset_days=0 as the default (None also means 0). The logic itself is the base
# function, unchanged; an explicit offset_days (positional or keyword) is passed through.
# ============================================================

def _base_fn(fn):
    return getattr(fn, "__wrapped__", fn)   # the original base function, even if this cell runs twice

_base_get_cohort_bounds   = _base_fn(get_cohort_bounds)
_base_get_cohort_months   = _base_fn(get_cohort_months)
_base_assign_cohort_month = _base_fn(assign_cohort_month)


def get_cohort_bounds(cohort_month, offset_days: int = 0):
    return _base_get_cohort_bounds(cohort_month, 0 if offset_days is None else offset_days)


def get_cohort_months(max_data_date=None, offset_days: int = 0):
    return _base_get_cohort_months(max_data_date, 0 if offset_days is None else offset_days)


def assign_cohort_month(svc_dt_col, offset_days: int = 0):
    return _base_assign_cohort_month(svc_dt_col, 0 if offset_days is None else offset_days)


for _w, _b in [(get_cohort_bounds, _base_get_cohort_bounds), (get_cohort_months, _base_get_cohort_months),
               (assign_cohort_month, _base_assign_cohort_month)]:
    _w.__wrapped__ = _b
    _w.__doc__ = (_b.__doc__ or "") + "\n    NBA 3.1: offset_days defaults to 0 (00_cohort_generation)."

In [0]:
from datetime import date, datetime, timezone
from dateutil.relativedelta import relativedelta
import json, os
import pandas as pd

# ============================================================
# COHORT SCHEDULE INPUTS (Excel NBA_3.1_Model_Architecture_and_Cohorts.xlsx, "Inputs")
# ============================================================
COHORT_1_PREDICTION_MONTH = date(2024, 7, 1)  # c1 prediction month
LOOKBACK_WINDOW_MONTHS = 12
DATA_LAG_MONTHS = 1                            # 4-week lag for claims
LAST_TRAINING_COHORT = 24                      # c1-c24
LAST_TESTING_COHORT = 26                       # c25-c26
LAST_HOLDOUT_COHORT = 28                       # c27-c28
TOTAL_COHORTS = 28

# ============================================================
# FEATURE-FAMILY ANCHOR OFFSETS
# ============================================================
CLAIMS_ANCHOR_OFFSET_DAYS = 0         # Rx, Dx, Px, Lab - 4-week lag, month-boundary
PROMOTIONAL_ANCHOR_OFFSET_DAYS = 17   # Calls, iRep, DTC, Digital, Samples, Speaker, FTP, HQ Emails, ClickStream - 2-week lag, day-17
MMIT_ANCHOR_OFFSET_DAYS = None        # TBD - must be set before use
LAAD_LRX_ANCHOR_OFFSET_DAYS = None    # TBD - must be set before use

# Where the cohorts are saved: <root>/<family>/cohorts.parquet and <root>/<family>/metadata.json
COHORT_VOLUME_ROOT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/cohorts"

In [0]:
_FAMILY_OFFSET_CONSTANTS = {
    "claims":      "CLAIMS_ANCHOR_OFFSET_DAYS",
    "promotional": "PROMOTIONAL_ANCHOR_OFFSET_DAYS",
    "mmit":        "MMIT_ANCHOR_OFFSET_DAYS",
    "laad_lrx":    "LAAD_LRX_ANCHOR_OFFSET_DAYS",
}

# c1 lookback dates from the Excel, used as known answers in validate(): family -> (offset, start, end)
_EXPECTED_C1_LOOKBACK = {
    "claims":      (0,  date(2023, 6, 1),  date(2024, 5, 1)),
    "promotional": (17, date(2023, 6, 17), date(2024, 6, 17)),
}

COHORT_COLUMNS = ["cohort_id", "prediction_month", "dataset", "offset_days",
                  "lookback_start_date", "lookback_end_date"]
_COHORT_SPARK_SCHEMA = ("cohort_id string, prediction_month date, dataset string, offset_days int, "
                        "lookback_start_date date, lookback_end_date date")


def _dataset_for(i):
    if i <= LAST_TRAINING_COHORT:
        return "Training"
    if i <= LAST_TESTING_COHORT:
        return "Testing"
    if i <= LAST_HOLDOUT_COHORT:
        return "Holdout (Backtesting)"
    raise ValueError(f"Cohort c{i} is beyond LAST_HOLDOUT_COHORT = {LAST_HOLDOUT_COHORT}")


class CohortGenerator:
    """NBA 3.1 cohort schedule (c1-c28) for one feature family.

    CohortGenerator(feature_family, offset_days=None)
        feature_family: 'claims', 'promotional', 'mmit' or 'laad_lrx'.
        offset_days:    optional override. By default the family's *_ANCHOR_OFFSET_DAYS constant
                        is used. If that constant is None, a ValueError is raised right away.

    Methods
        generate() -> pd.DataFrame   build and validate the 28 cohorts. Columns: cohort_id,
                                     prediction_month, dataset, offset_days, lookback_start_date,
                                     lookback_end_date (dates as datetime.date).
        validate(df)                 the checks run before saving. Raises ValueError listing what diverged.
        save(df=None) -> str         validate, then write <root>/<family>/cohorts.parquet (Spark) and
                                     metadata.json. Returns the parquet path. df=None generates first.
        load() -> pd.DataFrame       read the saved cohorts (no regeneration). Checks them against
                                     metadata.json and the current offset.
        metadata() -> dict           the saved metadata.json.

    Lookback formulas
        claims:      end = prediction_month - (DATA_LAG_MONTHS + 1) months (day 1)
                     start = end - (LOOKBACK_WINDOW_MONTHS - 1) months (day 1)
        promotional: end = day <offset> of the month before prediction_month
                     start = end - LOOKBACK_WINDOW_MONTHS months (same day)
        mmit, laad_lrx: not defined yet (NotImplementedError).
    """

    FAMILIES = tuple(_FAMILY_OFFSET_CONSTANTS)

    def __init__(self, feature_family: str, offset_days: int = None):
        family = str(feature_family).strip().lower()
        if family not in _FAMILY_OFFSET_CONSTANTS:
            raise ValueError(f"Unknown feature_family {feature_family!r}. Expected one of {list(self.FAMILIES)}.")
        const = _FAMILY_OFFSET_CONSTANTS[family]
        off = offset_days if offset_days is not None else globals().get(const)
        if off is None:
            raise ValueError(f"{const} is not set. Set it at the top of 00_cohort_generation "
                             f"or pass offset_days= to CohortGenerator().")
        if isinstance(off, bool) or not isinstance(off, int) or not (0 <= off <= 28):
            raise ValueError(f"{family}: offset_days must be an int day of month between 0 and 28, got {off!r}")
        self.feature_family = family
        self.offset_constant = const
        self.offset_days = off
        self.dir = f"{COHORT_VOLUME_ROOT}/{family}"
        self.path = f"{self.dir}/cohorts.parquet"
        self.metadata_path = f"{self.dir}/metadata.json"

    def __repr__(self):
        return f"CohortGenerator('{self.feature_family}', offset_days={self.offset_days})"

    # ---- schedule ----
    def _lookback(self, prediction_month):
        if self.feature_family == "claims":
            day = self.offset_days if self.offset_days > 0 else 1        # offset 0 = month boundary (day 1)
            end = (prediction_month - relativedelta(months=DATA_LAG_MONTHS + 1)).replace(day=day)
            start = end - relativedelta(months=LOOKBACK_WINDOW_MONTHS - 1)
        elif self.feature_family == "promotional":
            end = (prediction_month - relativedelta(months=1)).replace(day=self.offset_days or 1)
            start = end - relativedelta(months=LOOKBACK_WINDOW_MONTHS)
        else:
            raise NotImplementedError(f"The lookback formula for '{self.feature_family}' is not defined yet. "
                                      f"Add it to CohortGenerator._lookback when {self.offset_constant} is chosen.")
        return start, end

    def generate(self) -> pd.DataFrame:
        rows = []
        for i in range(1, TOTAL_COHORTS + 1):
            pm = COHORT_1_PREDICTION_MONTH + relativedelta(months=i - 1)
            start, end = self._lookback(pm)
            rows.append((f"c{i}", pm, _dataset_for(i), self.offset_days, start, end))
        df = pd.DataFrame(rows, columns=COHORT_COLUMNS)
        self.validate(df)
        return df

    # ---- checks ----
    def validate(self, df: pd.DataFrame):
        errors = []
        if TOTAL_COHORTS != LAST_HOLDOUT_COHORT:
            errors.append(f"TOTAL_COHORTS ({TOTAL_COHORTS}) != LAST_HOLDOUT_COHORT ({LAST_HOLDOUT_COHORT})")
        if len(df) != 28 or len(df) != TOTAL_COHORTS:
            errors.append(f"expected exactly 28 rows (the Excel cohort schedule), got {len(df)}")
        missing = [c for c in COHORT_COLUMNS if c not in df.columns]
        if missing:
            errors.append(f"missing columns {missing}")
            raise ValueError(f"{self}: cohort validation failed:\n  - " + "\n  - ".join(errors))
        ids = list(df["cohort_id"])
        exp_ids = [f"c{i}" for i in range(1, TOTAL_COHORTS + 1)]
        if ids != exp_ids:
            errors.append(f"cohort_id must be c1..c{TOTAL_COHORTS} in order, got {ids[:3]}...{ids[-3:]}")
        counts = df["dataset"].value_counts().to_dict()
        exp_counts = {"Training": LAST_TRAINING_COHORT,
                      "Testing": LAST_TESTING_COHORT - LAST_TRAINING_COHORT,
                      "Holdout (Backtesting)": LAST_HOLDOUT_COHORT - LAST_TESTING_COHORT}
        if counts != exp_counts:
            errors.append(f"dataset split should be {exp_counts}, got {counts}")
        if list(df["dataset"]) != [_dataset_for(i) for i in range(1, len(df) + 1)]:
            errors.append("dataset labels are not in cohort order (Training, then Testing, then Holdout)")
        exp_pm = [COHORT_1_PREDICTION_MONTH + relativedelta(months=i) for i in range(len(df))]
        if list(df["prediction_month"]) != exp_pm:
            errors.append(f"prediction_month must run monthly from {COHORT_1_PREDICTION_MONTH}")
        if set(df["offset_days"]) != {self.offset_days}:
            errors.append(f"offset_days should all be {self.offset_days}, got {sorted(set(df['offset_days']))}")
        if not (df["lookback_start_date"] < df["lookback_end_date"]).all():
            errors.append("lookback_start_date must be before lookback_end_date in every cohort")
        if not (df["lookback_end_date"] < df["prediction_month"]).all():
            errors.append("lookback_end_date must be before prediction_month in every cohort")
        exp = _EXPECTED_C1_LOOKBACK.get(self.feature_family)
        if exp and exp[0] == self.offset_days and len(df):
            got = (df.loc[0, "lookback_start_date"], df.loc[0, "lookback_end_date"])
            if got != exp[1:]:
                errors.append(f"c1 lookback should be {exp[1]} -> {exp[2]} (Excel), got {got[0]} -> {got[1]}")
        if errors:
            raise ValueError(f"{self}: cohort validation failed:\n  - " + "\n  - ".join(errors))

    # ---- persistence ----
    def save(self, df: pd.DataFrame = None) -> str:
        if df is None:
            df = self.generate()
        self.validate(df)
        records = [(r.cohort_id, r.prediction_month, r.dataset, int(r.offset_days),
                    r.lookback_start_date, r.lookback_end_date) for r in df.itertuples(index=False)]
        (spark.createDataFrame(records, _COHORT_SPARK_SCHEMA)
              .coalesce(1).write.mode("overwrite").parquet(self.path))
        meta = {
            "feature_family": self.feature_family,
            "offset_constant": self.offset_constant,
            "offset_days": self.offset_days,
            "generation_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "cohort_count": len(df),
            "lookback_window_months": LOOKBACK_WINDOW_MONTHS,
            "data_lag_months": DATA_LAG_MONTHS,
            "cohort_1_prediction_month": str(COHORT_1_PREDICTION_MONTH),
            "first_cohort": f"{df.iloc[0]['cohort_id']} {df.iloc[0]['prediction_month']}",
            "last_cohort": f"{df.iloc[-1]['cohort_id']} {df.iloc[-1]['prediction_month']}",
            "path": self.path,
        }
        os.makedirs(self.dir, exist_ok=True)
        with open(self.metadata_path, "w") as fh:
            json.dump(meta, fh, indent=2)
        return self.path

    def metadata(self) -> dict:
        if not os.path.exists(self.metadata_path):
            raise FileNotFoundError(f"No metadata for '{self.feature_family}' at {self.metadata_path}. "
                                    f"Run 00_cohort_generation end to end first.")
        with open(self.metadata_path) as fh:
            return json.load(fh)

    def load(self) -> pd.DataFrame:
        if not os.path.exists(self.path):
            raise FileNotFoundError(f"No saved cohorts for '{self.feature_family}' at {self.path}. Run "
                                    f"00_cohort_generation end to end (or CohortGenerator('{self.feature_family}').save()) first.")
        meta = self.metadata()
        if meta.get("offset_days") != self.offset_days:
            raise ValueError(f"Saved '{self.feature_family}' cohorts were built at offset {meta.get('offset_days')}, "
                             f"but {self.offset_constant} is now {self.offset_days}. Re-run 00_cohort_generation.")
        df = spark.read.parquet(self.path).toPandas()
        df = (df.assign(_n=df["cohort_id"].str[1:].astype(int)).sort_values("_n")
                .drop(columns="_n").reset_index(drop=True)[COHORT_COLUMNS])
        df["offset_days"] = df["offset_days"].astype("int64")      # same dtype as generate()
        if len(df) != meta.get("cohort_count"):
            raise ValueError(f"{self.path} has {len(df)} rows but metadata.json says {meta.get('cohort_count')}")
        df.attrs["metadata"] = meta
        return df

In [0]:
# Runs only when THIS notebook is executed directly (interactively or as a job task).
# When another notebook %runs it, the notebook path in the context is the caller's, so nothing is regenerated.
_GENERATION_NOTEBOOK_NAME = "00_cohort_generation"


def _top_level_notebook_name():
    try:
        p = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return p.rstrip("/").split("/")[-1]
    except Exception:
        return None


def generate_all_cohorts():
    """Generate, validate and save every family whose offset is set. Returns {family: path}."""
    written = {}
    for family, const in _FAMILY_OFFSET_CONSTANTS.items():
        if globals().get(const) is None:
            print(f"SKIPPED {family} — {const} is None")
            continue
        gen = CohortGenerator(family)
        df = gen.generate()
        path = gen.save(df)
        written[family] = path
        print(f"GENERATED {family} — offset={gen.offset_days}, rows={len(df)}, path={path}")
    return written


if _top_level_notebook_name() == _GENERATION_NOTEBOOK_NAME:
    _written = generate_all_cohorts()
else:
    print(f"00_cohort_generation loaded by '{_top_level_notebook_name()}': no cohorts regenerated. "
          f"Use CohortGenerator('<family>').load().")